# 08 — Bridge Type Decision Engine — Independent Validation

## Objective

Validate the two ML components used by the bridge-type decision engine on an independent bridge-level test set:

1. `Bauwerksart` classification from the four agreed project inputs.
2. `zustandsnote` estimation using the four project inputs plus candidate `Bauwerksart`.

This notebook does not create a structural design and does not use FEM/InfoCAD.

The purpose is independent validation of the decision-engine components before the explainability/robustness and final engineering-decision stages.


In [ ]:
from getpass import getpass
from pathlib import Path
import os
import json
import warnings

import numpy as np
import pandas as pd

from sqlalchemy import create_engine, URL, text
from pyproj import Transformer

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    top_k_accuracy_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    confusion_matrix,
)
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

def find_project_root():
    env_root = os.getenv("BRIDGE_PROJECT_ROOT")
    if env_root:
        root = Path(env_root).expanduser().resolve()
        if (root / "Dataset_PlanA-B").exists():
            return root
        raise FileNotFoundError(
            f"BRIDGE_PROJECT_ROOT does not contain Dataset_PlanA-B: {root}"
        )

    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "Dataset_PlanA-B").exists():
            return candidate

    raise FileNotFoundError(
        "Project root not found. Set BRIDGE_PROJECT_ROOT to the project folder."
    )

PROJECT_ROOT = find_project_root()
DATASET_ROOT = PROJECT_ROOT / "Dataset_PlanA-B"
OUTPUT_ROOT = PROJECT_ROOT / "Output_PlanA-B"
OUTPUT_DIR = OUTPUT_ROOT / "08_Bridge_Type_Decision_Engine_Independent_Validation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "Final_Project"
DB_USER = "postgres"
DB_PASSWORD = getpass("PostgreSQL password: ")

print("Project root:", PROJECT_ROOT)
print("Dataset root:", DATASET_ROOT)
print("Output dir :", OUTPUT_DIR)
print("Notebook stage: 08 — Independent Validation")


## 00A — DATA SOURCE / INPUT–OUTPUT MANIFEST

| Item | Source / origin | Transfer method | Role in Notebook 08 | Destination |
|---|---|---|---|---|
| ML dataset | PostgreSQL `Final_Project` → `final.bridge_ml_dataset_final` | SQL query | Canonical independent-validation input | In-memory `df` |
| Geometry | `geom_x` / `geom_y` | SQL + EPSG:3857 → EPSG:4326 | Latitude/longitude predictors | `work` |
| Traffic | `traffic_dtv_mean` | SQL | DTV predictor | `work` |
| Material | `baustoffklasse` | SQL | Material predictor | `work` |
| Bridge type | `bauwerksart_text` | SQL | Classification target / condition predictor | `work` |
| Condition | `zustandsnote` | SQL | Condition target | `work` |
| Validation results | Model/dataframe results | Local file write | Independent validation package | `Output_PlanA-B/08_Bridge_Type_Decision_Engine_Independent_Validation` |

### Transfer chain

```text
Notebook 06
    ↓
PostgreSQL: final.bridge_ml_dataset_final
    ↓
Notebook 08
    ├── independent bridge-level split
    ├── Bauwerksart classifier validation
    ├── condition-model validation
    └── counterfactual candidate-type simulation
    ↓
Output_PlanA-B/08_Bridge_Type_Decision_Engine_Independent_Validation
```

**No BASt/DWD/Traffic download occurs in Notebook 08.**

**No imputation pipeline is performed here; the notebook consumes the canonical dataset already loaded into PostgreSQL by Notebook 06.**

**The counterfactual candidate-type simulation has no observed ground truth for alternative bridge types. It is therefore reported as a model experiment, not as a validated structural/design outcome.**

**No FEM/InfoCAD calculation is performed.**


## 01 — Load canonical dataset


In [ ]:
# 01 — Load canonical PostgreSQL dataset

SOURCE_TABLE = '"final"."bridge_ml_dataset_final"'

url = URL.create(
    "postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)

engine = create_engine(url, connect_args={"connect_timeout": 10})

with engine.connect() as conn:
    print("PostgreSQL connection: PASS")
    print("Database:", conn.execute(text("SELECT current_database()")).scalar())

df = pd.read_sql(
    f"SELECT * FROM {SOURCE_TABLE}",
    engine,
)

print("Source table:", SOURCE_TABLE)
print("Shape:", df.shape)
print("Columns:", len(df.columns))


## 02 — Prepare canonical inputs


In [ ]:
required = [
    "bridge_id",
    "geom_x",
    "geom_y",
    "traffic_dtv_mean",
    "baustoffklasse",
    "bauwerksart_text",
    "zustandsnote",
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise RuntimeError(f"Missing required columns: {missing}")

tr = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)

x = pd.to_numeric(df["geom_x"], errors="coerce")
y = pd.to_numeric(df["geom_y"], errors="coerce")
longitude, latitude = tr.transform(
    x.to_numpy(dtype=float),
    y.to_numpy(dtype=float),
)

work = pd.DataFrame({
    "bridge_id": df["bridge_id"].astype(str),
    "latitude": latitude,
    "longitude": longitude,
    "dtv": pd.to_numeric(df["traffic_dtv_mean"], errors="coerce"),
    "bauwerkstoff": df["baustoffklasse"].astype("string").str.strip(),
    "bauwerksart": df["bauwerksart_text"].astype("string").str.strip(),
    "zustandsnote": pd.to_numeric(df["zustandsnote"], errors="coerce"),
}).dropna()

work = work[
    work["zustandsnote"].between(1, 4)
    & work["latitude"].between(-90, 90)
    & work["longitude"].between(-180, 180)
].copy()

print("Usable rows:", f"{len(work):,}")
print("Bridge types:", work["bauwerksart"].nunique())


## 03 — Independent bridge-level split

The same bridge must never occur in both training and test data.


In [ ]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

train_idx, test_idx = next(
    gss.split(
        work,
        work["zustandsnote"],
        groups=work["bridge_id"],
    )
)

train = work.iloc[train_idx].copy()
test = work.iloc[test_idx].copy()

print("Train rows:", len(train))
print("Test rows:", len(test))
print("Train bridges:", train["bridge_id"].nunique())
print("Test bridges:", test["bridge_id"].nunique())

overlap = set(train["bridge_id"]) & set(test["bridge_id"])
print("Bridge-ID overlap:", len(overlap))
assert len(overlap) == 0


## 04 — Define stable candidate types

The candidate universe is determined from the training population only.


In [ ]:
MIN_HISTORICAL_N = 100

train_type_counts = train["bauwerksart"].value_counts()
candidate_types = train_type_counts[
    train_type_counts >= MIN_HISTORICAL_N
].index.tolist()

test_candidate = test[
    test["bauwerksart"].isin(candidate_types)
].copy()

print("Candidate types:", len(candidate_types))
print("Test rows with candidate types:", len(test_candidate))


## 05 — Validate Bauwerksart classifier


In [ ]:
CLS_FEATURES = ["latitude", "longitude", "dtv", "bauwerkstoff"]

cls_pre = ColumnTransformer([
    (
        "num",
        SimpleImputer(strategy="median"),
        ["latitude", "longitude", "dtv"],
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]),
        ["bauwerkstoff"],
    ),
])

classifier = Pipeline([
    ("preprocess", cls_pre),
    ("model", ExtraTreesClassifier(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced",
    )),
])

classifier.fit(
    train[CLS_FEATURES],
    train["bauwerksart"],
)

cls_pred = classifier.predict(test[CLS_FEATURES])
cls_prob = classifier.predict_proba(test[CLS_FEATURES])
cls_classes = classifier.named_steps["model"].classes_

top1 = accuracy_score(test["bauwerksart"], cls_pred)
balanced = balanced_accuracy_score(test["bauwerksart"], cls_pred)

k = min(5, len(cls_classes))
top5 = top_k_accuracy_score(
    test["bauwerksart"],
    cls_prob,
    k=k,
    labels=cls_classes,
)

classification_metrics = pd.DataFrame([{
    "accuracy_top1": top1,
    "balanced_accuracy": balanced,
    f"accuracy_top{k}": top5,
    "n_test": len(test),
    "n_classes": len(cls_classes),
}])

display(classification_metrics)


## 06 — Classifier performance by bridge type

This identifies classes where the classifier has insufficient evidence.


In [ ]:
cls_test = test[["bridge_id", "bauwerksart"]].copy()
cls_test["predicted_bauwerksart"] = cls_pred
cls_test["correct"] = (
    cls_test["bauwerksart"] ==
    cls_test["predicted_bauwerksart"]
)

classification_by_type = (
    cls_test.groupby("bauwerksart")
    .agg(
        n=("bridge_id", "size"),
        accuracy=("correct", "mean"),
    )
    .sort_values("n", ascending=False)
)

display(classification_by_type)


## 07 — Validate `zustandsnote` model

The model sees the actual observed `Bauwerksart` for each test bridge.

This is a predictive validation of condition estimation, not a counterfactual validation of an alternative bridge type.


In [ ]:
REG_FEATURES = [
    "latitude",
    "longitude",
    "dtv",
    "bauwerkstoff",
    "bauwerksart",
]

print("REG_FEATURES:", REG_FEATURES)


In [ ]:
reg_pre = ColumnTransformer([
    (
        "num",
        SimpleImputer(strategy="median"),
        ["latitude", "longitude", "dtv"],
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]),
        ["bauwerkstoff", "bauwerksart"],
    ),
])

condition_model = Pipeline([
    ("preprocess", reg_pre),
    ("model", ExtraTreesRegressor(
        n_estimators=500,
        min_samples_leaf=5,
        max_features=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

condition_model.fit(
    train[REG_FEATURES],
    train["zustandsnote"],
)

condition_pred = np.clip(
    condition_model.predict(test[REG_FEATURES]),
    1,
    4,
)

condition_metrics = pd.DataFrame([{
    "MAE": mean_absolute_error(test["zustandsnote"], condition_pred),
    "RMSE": mean_squared_error(test["zustandsnote"], condition_pred) ** 0.5,
    "R2": r2_score(test["zustandsnote"], condition_pred),
    "n_test": len(test),
}])

display(condition_metrics)


## 08 — Condition error by bridge type


In [ ]:
condition_test = test[
    ["bridge_id", "bauwerksart", "zustandsnote"]
].copy()

condition_test["predicted_zustandsnote"] = condition_pred
condition_test["absolute_error"] = (
    condition_test["zustandsnote"] -
    condition_test["predicted_zustandsnote"]
).abs()

condition_by_type = (
    condition_test.groupby("bauwerksart")
    .agg(
        n=("bridge_id", "size"),
        MAE=("absolute_error", "mean"),
        observed_mean=("zustandsnote", "mean"),
        predicted_mean=("predicted_zustandsnote", "mean"),
    )
    .sort_values("n", ascending=False)
)

display(condition_by_type)


## 09 — Check the intended decision logic

For each test bridge, simulate the decision engine with the bridge's observed:

- location
- DTV
- Bauwerkstoff

Then calculate condition estimates for all candidate bridge types.

This is a **counterfactual model experiment**. It is not treated as ground truth because the real bridge was built with only one observed type.


In [ ]:
def candidate_condition_predictions(
    model,
    latitude,
    longitude,
    dtv,
    bauwerkstoff,
    candidates,
):
    x = pd.DataFrame({
        "latitude": latitude,
        "longitude": longitude,
        "dtv": dtv,
        "bauwerkstoff": bauwerkstoff,
        "bauwerksart": candidates,
    })
    pred = np.clip(
        model.predict(x[REG_FEATURES]),
        1,
        4,
    )
    x["predicted_zustandsnote"] = pred
    return x

sample = test_candidate.head(min(100, len(test_candidate))).copy()

simulation_rows = []

for _, row in sample.iterrows():
    cand = candidate_condition_predictions(
        condition_model,
        row["latitude"],
        row["longitude"],
        row["dtv"],
        row["bauwerkstoff"],
        candidate_types,
    )

    best_idx = cand["predicted_zustandsnote"].idxmin()
    selected_type = cand.loc[best_idx, "bauwerksart"]

    simulation_rows.append({
        "bridge_id": row["bridge_id"],
        "observed_bauwerksart": row["bauwerksart"],
        "selected_by_condition_model": selected_type,
        "observed_zustandsnote": row["zustandsnote"],
    })

simulation = pd.DataFrame(simulation_rows)

display(simulation.head(20))


## 10 — Important interpretation check

There is no valid accuracy metric for the counterfactual selected type because the dataset does not contain the condition that the same bridge would have had if a different `Bauwerksart` had been built.

Therefore this notebook does **not** claim that the selected counterfactual type is objectively the best bridge type.

The defensible validation consists of:

- classifier performance on observed `Bauwerksart`
- condition-model performance on observed `zustandsnote`
- historical condition evidence by bridge type
- transparent counterfactual estimates for engineering review


In [ ]:
selection_summary = pd.DataFrame([{
    "classifier_top1_accuracy": top1,
    f"classifier_top{k}_accuracy": top5,
    "classifier_balanced_accuracy": balanced,
    "condition_MAE": condition_metrics.loc[0, "MAE"],
    "condition_RMSE": condition_metrics.loc[0, "RMSE"],
    "condition_R2": condition_metrics.loc[0, "R2"],
    "candidate_type_count": len(candidate_types),
}])

display(selection_summary)


## 11 — Export validation package


In [ ]:
# 11 — Export independent validation package

classification_metrics.to_csv(
    OUTPUT_DIR / "08_classifier_validation_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

classification_by_type.to_csv(
    OUTPUT_DIR / "08_classifier_validation_by_type.csv",
    encoding="utf-8-sig",
)

condition_metrics.to_csv(
    OUTPUT_DIR / "08_condition_validation_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

condition_by_type.to_csv(
    OUTPUT_DIR / "08_condition_validation_by_type.csv",
    encoding="utf-8-sig",
)

simulation.to_csv(
    OUTPUT_DIR / "08_counterfactual_selection_simulation.csv",
    index=False,
    encoding="utf-8-sig",
)

selection_summary.to_csv(
    OUTPUT_DIR / "08_validation_summary.csv",
    index=False,
    encoding="utf-8-sig",
)

# Actual analysis-table inventory
data_inventory = pd.DataFrame({
    "column_order": range(1, len(work.columns) + 1),
    "column": work.columns,
    "dtype": [str(work[c].dtype) for c in work.columns],
    "missing_count": [int(work[c].isna().sum()) for c in work.columns],
    "role": [
        "group_id" if c == "bridge_id" else
        "classification_target" if c == "bauwerksart" else
        "condition_target" if c == "zustandsnote" else
        "predictor" if c in ["latitude", "longitude", "dtv", "bauwerkstoff"] else
        "derived_or_validation"
        for c in work.columns
    ],
})
data_inventory.to_csv(
    OUTPUT_DIR / "08_analysis_data_inventory.csv",
    index=False,
    encoding="utf-8-sig",
)

manifest = {
    "stage": 8,
    "notebook": "08_Bridge_Type_Decision_Engine_Independent_Validation",
    "source": SOURCE_TABLE,
    "validation_type": "independent bridge-level test validation",
    "new_project_inputs": [
        "latitude",
        "longitude",
        "dtv",
        "bauwerkstoff",
    ],
    "classification_target": "bauwerksart",
    "condition_target": "zustandsnote",
    "excluded": [
        "laenge",
        "breite",
        "FEM",
        "InfoCAD",
    ],
    "candidate_universe_rule": "training population only; minimum historical n >= 100",
    "counterfactual_selection": True,
    "counterfactual_ground_truth_available": False,
    "random_state": RANDOM_STATE,
    "outputs": [
        "08_classifier_validation_metrics.csv",
        "08_classifier_validation_by_type.csv",
        "08_condition_validation_metrics.csv",
        "08_condition_validation_by_type.csv",
        "08_counterfactual_selection_simulation.csv",
        "08_validation_summary.csv",
        "08_analysis_data_inventory.csv",
        "08_data_manifest.json",
        "08_data_manifest.txt",
    ],
}

(OUTPUT_DIR / "08_data_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

human_manifest = (
    "Notebook 08 — Bridge Type Decision Engine — Independent Validation\n"
    "===================================================================\n\n"
    f"PROJECT_ROOT: {PROJECT_ROOT}\n"
    f"SOURCE: PostgreSQL {SOURCE_TABLE}\n"
    f"OUTPUT_DIR: {OUTPUT_DIR}\n\n"
    "INPUTS:\n"
    "- latitude\n"
    "- longitude\n"
    "- dtv\n"
    "- bauwerkstoff\n\n"
    "TARGETS:\n"
    "- bauwerksart (classification)\n"
    "- zustandsnote (condition regression)\n\n"
    "EXCLUDED FROM THE MODELS:\n"
    "- laenge\n"
    "- breite\n"
    "- FEM\n"
    "- InfoCAD\n\n"
    "VALIDATION:\n"
    "- GroupShuffleSplit by bridge_id\n"
    "- Candidate types determined from training population only\n"
    "- Minimum historical candidate count: 100\n"
    "- Counterfactual alternative-type ground truth is unavailable\n\n"
    "OUTPUTS:\n"
    "- 08_classifier_validation_metrics.csv\n"
    "- 08_classifier_validation_by_type.csv\n"
    "- 08_condition_validation_metrics.csv\n"
    "- 08_condition_validation_by_type.csv\n"
    "- 08_counterfactual_selection_simulation.csv\n"
    "- 08_validation_summary.csv\n"
    "- 08_analysis_data_inventory.csv\n"
    "- 08_data_manifest.json\n"
    "- 08_data_manifest.txt\n"
)

(OUTPUT_DIR / "08_data_manifest.txt").write_text(
    human_manifest,
    encoding="utf-8",
)

print("08 STATUS: COMPLETE")
print("Output directory:", OUTPUT_DIR)
print("Data manifest:", OUTPUT_DIR / "08_data_manifest.txt")
